# webbot 全自動パイプライン(Kaggle)

データの準備 → 事前学習 → 検索の索引 → SFTデータ → SFTと評価、を1回の実行で最後まで進めます。**「Save & Run All」で実行すれば、ブラウザもPCも閉じて大丈夫です。**

## 実行のしかた
1. 右側の設定(Session options)で、**Accelerator** を「GPU T4 x2」か「GPU P100」に、**Internet** を「On」にします(使うGPUは1枚です。GPUとインターネットには電話番号の確認が必要です)
2. 右上の **Save Version** →「Save & Run All (Commit)」→ **Save** を押します
3. あとは閉じてかまいません。全部で7〜8時間の見込みです(全データで約1周の事前学習が約6時間)

## 結果
終わったら、このノートブックの **Output** に次のものが出ます。
- `pipeline.log`: 各段階の記録
- `runs/base/train.log`: 事前学習の損失の推移と生成文
- `runs/sft/train.log`: SFTの損失の推移と、評価用の質問(事前学習でもSFTでも見ていない記事についての質問)での答えの完全一致の割合
- `runs/base/checkpoint.pt`, `runs/sft/checkpoint.pt`, `data/tokenizer.json`: モデルとトークナイザー

## 途中で止まったとき(12時間の上限など)
1. 右側の **Add Input** →「Your Work」→ このノートブックの出力を追加します
2. もう一度「Save & Run All」します。追加した出力を自動で見つけて、済んだ段階は飛ばし、学習はチェックポイントの続きから再開します

## 1. 設定

In [ ]:
TRAIN_TOKENS = 800_000_000   # 事前学習で読むトークン数(全データ約8億で約1周)
SFT_STEPS = 6000
DEADLINE_HOURS = 11.0        # Kaggleの1回12時間の上限より前に、学習を止めて出力を保存する

import glob, os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
# 前回の出力を Add Input で追加していれば、その続きから進める
previous = sorted((os.path.dirname(p) for p in glob.glob("/kaggle/input/**/pipeline.log", recursive=True)),
                  key=lambda d: d.count("/"))  # the shallowest is the run's own
RESUME_FROM = previous[0] if previous else None
print("resume from:", RESUME_FROM)

## 2. GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 3. コードの取得とインストール

Rust版のBPE(トークナイザーの学習とエンコードが約25倍速い)もビルドします。

In [ ]:
CODE = "/tmp/webbot_code"
if not os.path.exists(CODE):
    !git clone -q https://github.com/KoRo2222/tiny-gpt-librarian {CODE}
!pip -q install maturin
if not os.path.exists(os.path.expanduser("~/.cargo/bin/cargo")):
    !curl -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal -q
os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + ":" + os.environ["PATH"]
!pip -q install {CODE}/rust
!python -c "import webbot_bpe; print('Rust BPE: ok')"

## 4. パイプライン

In [ ]:
cmd = (f"python {CODE}/scripts/run_pipeline.py --work /kaggle/working --scratch /tmp/webbot "
       f"--train-tokens {TRAIN_TOKENS} --sft-steps {SFT_STEPS} --deadline-hours {DEADLINE_HOURS}")
if RESUME_FROM:
    cmd += f" --resume-from {RESUME_FROM}"
!{cmd}

## 5. 結果

In [ ]:
!tail -n 5 /kaggle/working/pipeline.log
!tail -n 12 /kaggle/working/runs/base/train.log
!tail -n 10 /kaggle/working/runs/sft/train.log